In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import re
import ast
import json
import os
from tqdm.notebook import tqdm  # or from tqdm import tqdm
import matplotlib.pyplot as plt


# Translation Table

NB: HERE WE WORK ONLY WITH ARTICLES WHICH HAVE FULL-TEXT -> CAN IMPACT THE NUMBER OF REPORTED ENTITIES. IF THEY HAD NO STUDIES WITH ANNOTATIONS, THEY ARE EXCLUDED

In [ ]:
old_ds_analysis = False
full_ds_analysis = True
neuro_ds_analysis_17 = False

In [ ]:
# datasets prepared in Translation_05_0_Prep_Pairs_for_Associations
# for the old analysis translation_table_drug_disease_OLD.csv
if old_ds_analysis:
    file_path = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_OLD.csv"
    dataset_name = "OLD dataset"
elif full_ds_analysis:
    file_path = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_FULL.csv"
    dataset_name = "FULL dataset"
elif neuro_ds_analysis_17:
    file_path = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_NEURO_17.csv"
    dataset_name = "NEURO_17 dataset"
else:
    file_path = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_drug_disease_NEURO.csv"
    dataset_name = "NEURO dataset"

print(f"\nLoading translation table: {dataset_name}")
print(f"File path: {file_path}")

translation_table = pd.read_csv(file_path)

print("\nData loaded successfully")
print(f"Rows: {translation_table.shape[0]:,}")
print(f"Columns: {translation_table.shape[1]}")
print(f"Shape: {translation_table.shape}")

print("\nPreview:")
translation_table.head()

In [ ]:
keep_linked_only = False
if keep_linked_only:
    # Filter to keep rows where both IDs are NOT -1
    translation_table = translation_table[
        (translation_table['merged_mondo_termid'] != "-1") & 
        (translation_table['merged_umls_termid'] != "-1")
    ]
    
    # Reset index for a clean dataframe
    translation_table = translation_table.reset_index(drop=True)

In [ ]:
n_diseases = translation_table["merged_mondo_label"].nunique()
n_drugs = translation_table["merged_umls_label"].nunique()

print("\n=== Entity coverage summary ===")
print(f"Total unique diseases: {n_diseases:,}")
print(f"Total unique drugs: {n_drugs:,}")


In [ ]:
translation_table.normalized_key.nunique()

### filer for translated

In [ ]:
translation_table["translation_status"] = np.where(
    (
        translation_table["at_least_one_phase4"]
        | translation_table["at_least_one_phase3_completed"]
        | translation_table["fda_AP"]
    ),
    "approved",
    "failed",
)
print("=== Translation status summary ===")
print(translation_table["translation_status"].value_counts())
print()

In [ ]:
df_translated = translation_table[translation_table['translation_status']=="approved"]
df_translated.shape

In [ ]:
df_translated[df_translated['min_trial_start_year']>2015].shape

In [ ]:
test_recent = df_translated[df_translated['min_trial_start_year'].isna()]
test_recent[test_recent['fda_earliest_year']>2015].shape

In [ ]:
df_translated['min_trial_start_year'].dropna().hist(bins=20)
plt.xlabel("Year")
plt.ylabel("Number of trials")
plt.title("Distribution of Trial Start Years")
plt.show()

In [ ]:
df_translated["min_relevant_clinical_year"] = (
    df_translated[
        [
            "min_phase_4_year",
            "trial_completion_year_first_completed_phase3",
            "fda_earliest_year",
        ]
    ]
    .min(axis=1, skipna=True)
)
df_translated["min_relevant_clinical_year"] = (
    df_translated["min_relevant_clinical_year"]
    .round()
    .astype("Int64")   # nullable integer dtype
)


In [ ]:
df_translated[
    df_translated[
        [
            "min_relevant_clinical_year",
           
        ]
    ].isna().any(axis=1)
]

In [ ]:
def to_list(x):
    if not isinstance(x, str) or not x.startswith('['):
        return x if isinstance(x, list) else []
    
    # Replace 'nan' with 'None' so the parser understands it
    cleaned_x = x.replace('nan', 'None')
    
    try:
        return ast.literal_eval(cleaned_x)
    except Exception as e:
        # If literal_eval still fails, try json.loads as a backup
        try:
            return json.loads(cleaned_x.replace("'", '"'))
        except:
            print(f"Failed to parse: {x[:50]}...") # Print first 50 chars of error
            return []

# Apply the fix
df_translated.loc[:, 'preclinical_doc_ids'] = df_translated['preclinical_doc_ids'].apply(to_list)
df_translated.loc[:, 'pub_year'] = df_translated['pub_year'].apply(to_list)

df_translated.head(2)

In [ ]:
df_translated[df_translated['normalized_key']=='type 2 diabetes mellitus <> metformin']

#### exclude studies after clinical

In [ ]:
df_translated['preclinical_ids_before_clinical'] = df_translated.apply(
    lambda row: [
        doc_id for doc_id, year in zip(row['preclinical_doc_ids'], row['pub_year']) 
        if pd.notnull(row['min_relevant_clinical_year']) and pd.notnull(year) and float(year) <= row['min_relevant_clinical_year']
    ], 
    axis=1
)
df_translated['preclinical_count_before_clinical'] = df_translated['preclinical_ids_before_clinical'].str.len()
df_translated.shape

In [ ]:
translated_pmids = list(set([
    pmid 
    for sublist in df_translated['preclinical_ids_before_clinical'] 
    for pmid in sublist
]))
len(translated_pmids)

In [ ]:
translated_pmids[:5]

### not translated (and not recently entered clinical)

In [ ]:
df_failed = translation_table[translation_table['translation_status']=="failed"]
df_failed.shape

In [ ]:
df_recent = df_failed[df_failed['min_trial_start_year']>=2015]
df_recent.shape

In [ ]:
df_failed = df_failed[df_failed['min_trial_start_year']<2015]
df_failed.shape

In [ ]:
df_failed['min_trial_start_year'].dropna().hist(bins=20)
plt.xlabel("Year")
plt.ylabel("Number of trials")
plt.title("Distribution of Trial Start Years")
plt.show()

In [ ]:
df_failed.loc[:, 'max_trial_start_year'] = pd.to_numeric(df_failed['max_trial_start_year'], errors='coerce')
df_failed.loc[:, 'preclinical_doc_ids'] = df_failed['preclinical_doc_ids'].apply(to_list)
df_failed.loc[:, 'pub_year'] = df_failed['pub_year'].apply(to_list)

In [ ]:
df_failed["min_relevant_clinical_year"] = (
    df_failed["max_trial_start_year"]
    .round()
    .astype("Int64")   # nullable integer dtype
)

In [ ]:
df_failed[
    df_failed[
        [
            "min_relevant_clinical_year",
           
        ]
    ].isna().any(axis=1)
]

#### exclude studies after clinical

In [ ]:
df_failed['preclinical_ids_before_latest_trial'] = df_failed.apply(
    lambda row: [
        doc_id for doc_id, year in zip(row['preclinical_doc_ids'], row['pub_year']) 
        if pd.notnull(row['max_trial_start_year']) and pd.notnull(year) and float(year) <= row['max_trial_start_year']
    ], 
    axis=1
)
df_failed['preclinical_count_before_latest_trial'] = df_failed['preclinical_ids_before_latest_trial'].str.len()
df_failed.shape

In [ ]:
failed_pmids = list(set([
    pmid 
    for sublist in df_failed['preclinical_ids_before_latest_trial'] 
    for pmid in sublist
]))
len(failed_pmids)

In [ ]:
data_for_timeline = pd.concat([df_failed, df_translated], axis=0)
suffix = dataset_name.split()[0] 
data_for_timeline.to_csv(f"/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/translation_table_for_timeline_{suffix}.csv", index=False)

# Get relevant preclinical studies

In [ ]:
def summarize_preclinical_retrieval(
    df_original,
    df_targets,
    df_retrieved,
    target_pmid_col="pmid",
    retrieved_pmid_col="PMID",
    key_col="normalized_key",
    relevant_preclin_cound_col="preclinical_count_before_clinical", #"preclinical_count_before_latest_trial",
    verbose=True,
):
    """
    Summarize retrieval coverage after joining target PMIDs to a preclinical dataset.
    """

    # --- PMID-level retrieval ---
    target_pmids = df_targets[target_pmid_col].nunique()
    retrieved_pmids = df_retrieved[retrieved_pmid_col].nunique()

    pct_pmids_retrieved = 100 * retrieved_pmids / target_pmids if target_pmids > 0 else 0.0

    # --- normalized_key-level coverage ---
    keys_before = set(df_targets[key_col].dropna())
    keys_after = set(df_retrieved[key_col].dropna())

    n_keys_before = len(keys_before)
    n_keys_after = len(keys_after)

    keys_with_no_articles = keys_before - keys_after
    n_keys_with_no_articles = len(keys_with_no_articles)

    pct_keys_retrieved = 100 * n_keys_after / n_keys_before if n_keys_before > 0 else 0.0
    pct_keys_missing = 100 * n_keys_with_no_articles / n_keys_before if n_keys_before > 0 else 0.0

    if verbose:
        print("\n" + "=" * 60)
        print("PRECLINICAL RETRIEVAL SUMMARY")
        print("=" * 60)

        # --- Target population ---
        print("\n[1] Target drug–disease pairs")
        target_entity_pairs = df_original[key_col].nunique()
        print(f"Total target pairs: {target_entity_pairs:,}")

        missing_before_clinical = (
            df_original[relevant_preclin_cound_col] == 0
        ).sum()
        print(
            f"Pairs with NO preclinical studies before clinical: "
            f"{missing_before_clinical:,} "
            f"({100 * missing_before_clinical / target_entity_pairs:.1f}%)"
        )
        print(
            f"Pairs with ≥1 preclinical study before clinical: "
            f"{target_entity_pairs - missing_before_clinical:,}"
        )
        
        missing_before_clinical_g1 = (
            df_original[relevant_preclin_cound_col] <= 1
        ).sum()
        print(
            f"Pairs with less than 2 preclinical studies before clinical: "
            f"{missing_before_clinical_g1:,} "
            f"({100 * missing_before_clinical_g1 / target_entity_pairs:.1f}%)"
        )
        print(
            f"Pairs with ≥2 preclinical study before clinical: "
            f"{target_entity_pairs - missing_before_clinical_g1:,}"
        )
        

        # --- PMID retrieval ---
        print("\n[2] PMID-level retrieval")
        print(
            f"Target PMIDs:    {target_pmids:,}\n"
            f"Retrieved PMIDs: {retrieved_pmids:,}\n"
            f"Retrieval rate:  {pct_pmids_retrieved:.1f}%"
        )

        # --- normalized_key coverage ---
        print("\n[3] normalized_key coverage (entity-level)")
        print(
            f"Keys with ≥1 retrieved article: {n_keys_after:,} / {n_keys_before:,} "
            f"({pct_keys_retrieved:.1f}%)"
        )
        print(
            f"Keys with NO retrieved articles: {n_keys_with_no_articles:,} / {n_keys_before:,} "
            f"({pct_keys_missing:.1f}%)"
        )

        print("=" * 60 + "\n")


    return {
        "target_pmids": target_pmids,
        "retrieved_pmids": retrieved_pmids,
        "pct_pmids_retrieved": pct_pmids_retrieved,
        "n_keys_before": n_keys_before,
        "n_keys_after": n_keys_after,
        "n_keys_with_no_articles": n_keys_with_no_articles,
        "pct_keys_retrieved": pct_keys_retrieved,
        "pct_keys_missing": pct_keys_missing,
        "keys_with_no_articles": keys_with_no_articles,
    }


In [ ]:
base_annotation_dir = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions"

df_path_current_dataset = f"{base_annotation_dir}/full_text_combined_all_annotations_metadata.csv" #full_text_combined_all_annotations.csv" older version with less studies 340380. 'disease_term_mondo_norm','drug_term_umls_norm'
# ,'animal_age'
preclin_dataset = pd.read_csv(df_path_current_dataset)[['PMID','title', 'rigor_blinding', 'rigor_randomization', 'rigor_welfare', 'sample_size', 'animal_species','animal_sex','animal_strain','animal_number', 'assay_type', 'first_author_country']]
preclin_dataset['animal_sex'] = preclin_dataset['animal_sex'].apply(lambda x:x.replace("sex-",""))
#primekg_cleaned = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/entities_drug_disease_preclin.csv")
#preclin_dataset = preclin_dataset.merge(primekg_cleaned[['PMID','merged_mondo_label','merged_umls_label']], on="PMID", how="left")

preclin_dataset.head()

In [ ]:
preclin_dataset.shape

### translated subset

In [ ]:
df_translated_rel = df_translated[['normalized_key', 'preclinical_ids_before_clinical','merged_mondo_label','merged_umls_label','preclinical_count_before_clinical']]

# Explode the list column into individual rows
df_translated_exploded = df_translated_rel.explode('preclinical_ids_before_clinical')
df_translated_exploded = df_translated_exploded.reset_index(drop=True)
df_translated_exploded = df_translated_exploded.rename(columns={'preclinical_ids_before_clinical': 'pmid'})

df_translated_exploded = df_translated_exploded.dropna(subset=['pmid'])
df_translated_exploded.head()

In [ ]:
df_translated_exploded[df_translated_exploded.normalized_key.str.contains("parkinson disease <> levodopa")]

In [ ]:
df_translated_exploded.normalized_key.nunique()

In [ ]:
filtered_preclin_df_translated = pd.merge(
    df_translated_exploded, 
    preclin_dataset, 
    left_on='pmid', 
    right_on='PMID', 
    how='inner'
)

# 3. Clean up
# Since we have both 'pmid' and 'PMID' columns now, we can drop the lowercase one
filtered_preclin_df_translated = filtered_preclin_df_translated.drop(columns=['pmid']).reset_index(drop=True)


In [ ]:
filtered_preclin_df_translated.head()

In [ ]:
stats = summarize_preclinical_retrieval(
    df_translated,
    df_translated_exploded,
    filtered_preclin_df_translated,
)


In [ ]:
filtered_preclin_df_translated['study_classification'] = "approved"

In [ ]:
filtered_preclin_df_translated[filtered_preclin_df_translated.normalized_key.str.contains("parkinson disease <> levodopa")]

### failed subset

In [ ]:
docs_col = "preclinical_ids_before_latest_trial"
df_failed_rel = df_failed[['normalized_key', docs_col, 'merged_mondo_label','merged_umls_label','preclinical_count_before_latest_trial']] #preclinical_doc_ids

# Explode the list column into individual rows
df_failed_exploded = df_failed_rel.explode(docs_col)
df_failed_exploded = df_failed_exploded.reset_index(drop=True)
df_failed_exploded = df_failed_exploded.rename(columns={docs_col: 'pmid', 'preclinical_count_before_latest_trial': 'preclinical_count_before_clinical'})

df_failed_exploded = df_failed_exploded.dropna(subset=['pmid'])
df_failed_exploded.head()

In [ ]:
filtered_preclin_df_failed = pd.merge(
    df_failed_exploded, 
    preclin_dataset, 
    left_on='pmid', 
    right_on='PMID', 
    how='inner'
)

# 3. Clean up
# Since we have both 'pmid' and 'PMID' columns now, we can drop the lowercase one
filtered_preclin_df_failed = filtered_preclin_df_failed.drop(columns=['pmid']).reset_index(drop=True)
filtered_preclin_df_failed['study_classification'] = "failed"

print(f"Total rows after joining with PMIDs: {len(filtered_preclin_df_failed)} / {filtered_preclin_df_failed.PMID.nunique()}")

In [ ]:
stats = summarize_preclinical_retrieval(
    df_failed,
    df_failed_exploded,
    filtered_preclin_df_failed,
    relevant_preclin_cound_col="preclinical_count_before_latest_trial"
)


In [ ]:
1206/2331

In [ ]:
884 + 826

In [ ]:

(884 + 826)/5127

### concat

In [ ]:
filtered_preclin_df_translated['PMID'].count(), filtered_preclin_df_failed['PMID'].count()

In [ ]:
filtered_preclin_df_translated['PMID'].nunique(), filtered_preclin_df_failed['PMID'].nunique()

In [ ]:
filtered_preclin_df_failed.shape, filtered_preclin_df_translated.shape

In [ ]:
# Append the translated rows to the failed rows
combined_preclin_df = pd.concat([filtered_preclin_df_failed, filtered_preclin_df_translated], axis=0)

# Highly recommended: Reset the index
combined_preclin_df = combined_preclin_df.reset_index(drop=True)

In [ ]:
combined_preclin_df.normalized_key.count(), combined_preclin_df.normalized_key.nunique()

In [ ]:
combined_preclin_df.shape

In [ ]:
# Count how many PMID values are repeated
duplicate_count = combined_preclin_df.duplicated(subset=['PMID']).sum()

print(f"Number of duplicate PMIDs: {duplicate_count}")

In [ ]:
combined_preclin_df.head()

In [ ]:
suffix = dataset_name.split()[0] 
combined_preclin_df.to_csv(f"out/interm_drug_disease_pairs/drug_disease_pairs_annotations_flat_{suffix}.csv",index=False)

In [ ]:
def get_unique_drug_disease(df):
    n_diseases = df["merged_mondo_label"].nunique()
    n_drugs = df["merged_umls_label"].nunique()
    
    print("\n=== Entity coverage summary ===")
    print(f"Total unique diseases: {n_diseases:,}")
    print(f"Total unique drugs: {n_drugs:,}")

In [ ]:
get_unique_drug_disease(filtered_preclin_df_translated)

In [ ]:
get_unique_drug_disease(filtered_preclin_df_failed)

In [ ]:
get_unique_drug_disease(combined_preclin_df)

# Compute translation stats

In [ ]:
def to_pct(val):
    if pd.isna(val):
        return np.nan
    # already numeric: assume 0–1 are proportions, >=1 are percent values
    if isinstance(val, (int, float)):
        return float(val) * 100 if 0 <= float(val) <= 1 else float(val)
    s = str(val).strip()
    # match "(40.0%)" inside text like "6 (40.0%)"
    m = re.search(r'\(([-+]?\d*\.?\d+)\s*%?\)', s)
    if m:
        return float(m.group(1))
    # match trailing "40%" or "40.0 %"
    m = re.search(r'([-+]?\d*\.?\d+)\s*%$', s)
    if m:
        return float(m.group(1))
    # fallback: try raw float
    try:
        return float(s)
    except ValueError:
        return np.nan

In [ ]:
_splitter = re.compile(r'[|,]')

def _tokens(s):
    if pd.isna(s) or not str(s).strip():
        return set()
    return {t.strip() for t in _splitter.split(str(s)) if t.strip()}

def _strip_prefix_lower(x):
    x = x.lower().strip()
    x = re.sub(r'^(sex|species|strain|assay|country|first_author_country)[\s\-_]*', '', x)
    return x


def _sex_norms(sex_str):
    toks = {_strip_prefix_lower(t) for t in _tokens(sex_str)}
    out = set()
    for t in toks:
        if 'both' in t:        out.add('both')
        elif 'male' in t:      out.add('male')
        if 'female' in t:    out.add('female')
        if 'not' in t and 'report' in t:
            out.add('not-reported')
        elif t:
            out.add(t)
    return out
    
def compute_flags(row):
    sexes = _sex_norms(row.get('animal_sex'))
    both_sexes = ('both' in sexes) or ('male' in sexes and 'female' in sexes)

    species = _clean_set(row.get('animal_species'), exclude={'other','not reported'})
    strains = _clean_set(row.get('animal_strain'),  exclude={'unlabeled','unlabelled', 'not reported'})
    assays  = _clean_set(row.get('assay_type'),     exclude={'unlabeled','unlabelled', 'not reported'})
    countries = _clean_set(row.get('first_author_country'), exclude={'unlabeled','unlabelled','not reported'})

    # Prefer numeric article totals if present (from rigor_by_drug merge)
    articles_total = row.get("articles_total")
    if pd.isna(articles_total):
        # fallback to parsing PMID string if that's what you have
        articles_total = len(_clean_set(row.get("PMID")))

    bl = int(row.get("articles_blinding", 0) or 0)
    ra = int(row.get("articles_randomization", 0) or 0)
    wf = int(row.get("articles_welfare", 0) or 0)

    return pd.Series({
        'tested_both_sexes': int(both_sexes),
        'species_unique': len(species),
        'species_ge2': int(len(species) >= 2),
        'strain_unique': len(strains),
        'strains_ge2': int(len(strains) >= 2),
        'assay_unique': len(assays),
        'assays_ge2': int(len(assays) >= 2),
        'country_unique': len(countries),
        'countries_ge2': int(len(countries) >= 2),

        # article totals
        'articles_total': int(articles_total),

        # rigor: counts of articles with the flag
        'articles_blinding': bl,
        'articles_randomization': ra,
        'articles_welfare': wf,

        # rigor: >=1 article per drug with the flag
        'blinding_ge1_article': int(bl >= 1),
        'randomization_ge1_article': int(ra >= 1),
        'welfare_ge1_article': int(wf >= 1),
    })

def fmt_count_pct(count, total):
    pct = (count / total * 100) if total > 0 else 0
    return f"{count} ({pct:.1f}%)"
    
def _clean_set(cat_str, exclude=None):
    exclude = exclude or set()
    toks = {_strip_prefix_lower(t) for t in _tokens(cat_str)}
    return {t for t in toks if t and t not in exclude}
    
def unique_concat(series):
    vals = set()
    for x in series.dropna():
        for t in _splitter.split(str(x)):
            if t.strip():
                vals.add(t.strip())
    return ', '.join(sorted(vals)) if vals else ''


def pick_rig_cols(columns):
    order = [('rigor_blinding_binary','rigor_blinding'),
             ('rigor_randomization_binary','rigor_randomization'),
             ('rigor_welfare_binary','rigor_welfare')]
    picked = []
    for a,b in order:
        picked.append(a if a in columns else (b if b in columns else None))
    return picked

def pct(n, d):
    return f"{(n/d*100):.1f}%" if d > 0 else "0.0%"


def rob_percents_per_subset(df_subset, pick_rig_cols_fn):
    """Compute % of studies (unique PMIDs) with each ROB flag."""
    rb_col, rr_col, rw_col = pick_rig_cols_fn(df_subset.columns)
    rig_cols = [c for c in [rb_col, rr_col, rw_col] if c]

    if not rig_cols:
        return {"%_rigor_blinding": "0.0%", "%_rigor_randomization": "0.0%", "%_rigor_welfare": "0.0%"}

    tmp = df_subset[["PMID"] + rig_cols].copy()
    for c in rig_cols:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce").fillna(0).astype(int).clip(0, 1)

    # One record per study: a study counts as 1 if ANY row says 1
    per_study = tmp.groupby("PMID", as_index=False)[rig_cols].max()
    n_studies = len(per_study)

    rb = per_study[rb_col].sum() if rb_col else 0
    rr = per_study[rr_col].sum() if rr_col else 0
    rw = per_study[rw_col].sum() if rw_col else 0

    return {
        "%_rigor_blinding": pct(rb, n_studies),
        "%_rigor_randomization": pct(rr, n_studies),
        "%_rigor_welfare": pct(rw, n_studies),
    }

def rob_pair_counts_per_subset(
    df_subset,
    pick_rig_cols_fn,
    pair_col="normalized_key",
):
    """
    Compute pair-level rigor:
    number (percent) of drug–disease pairs where ANY supporting study reports each ROB item.
    """
    rb_col, rr_col, rw_col = pick_rig_cols_fn(df_subset.columns)
    rig_cols = [c for c in [rb_col, rr_col, rw_col] if c]

    if not rig_cols:
        return {
            "rigor_blinding_pairs": "0 (0.0%)",
            "rigor_randomization_pairs": "0 (0.0%)",
            "rigor_welfare_pairs": "0 (0.0%)",
        }

    tmp = df_subset[[pair_col] + rig_cols].copy()
    for c in rig_cols:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce").fillna(0).astype(int).clip(0, 1)

    # One record per pair: a pair counts as 1 if ANY row says 1
    per_pair = tmp.groupby(pair_col, as_index=False)[rig_cols].max()
    n_pairs = len(per_pair)

    rb = int(per_pair[rb_col].sum()) if rb_col else 0
    rr = int(per_pair[rr_col].sum()) if rr_col else 0
    rw = int(per_pair[rw_col].sum()) if rw_col else 0

    return {
        "rigor_blinding_pairs": fmt_count_pct(rb, n_pairs),
        "rigor_randomization_pairs": fmt_count_pct(rr, n_pairs),
        "rigor_welfare_pairs": fmt_count_pct(rw, n_pairs),
    }

def rigor_article_counts_by_entity(df, drug_col_name, pick_rig_cols_fn, study_id_col="PMID"):
    """
    For each drug, count UNIQUE articles (PMIDs) that report each rigor item.
    A PMID counts if ANY row for that (drug, PMID) has the rigor flag == 1.
    """
    rb_col, rr_col, rw_col = pick_rig_cols_fn(df.columns)
    rig_cols = [c for c in [rb_col, rr_col, rw_col] if c]

    # Base output with total articles per drug
    out = (
        df.groupby(drug_col_name, dropna=False)[study_id_col]
          .nunique()
          .rename("articles_total")
          .reset_index()
    )

    if not rig_cols:
        # no rigor columns available
        out["articles_blinding"] = 0
        out["articles_randomization"] = 0
        out["articles_welfare"] = 0
        return out

    tmp = df[[drug_col_name, study_id_col] + rig_cols].copy()
    for c in rig_cols:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce").fillna(0).astype(int).clip(0, 1)

    # one record per (drug, PMID): "any row says 1"
    per_drug_pmid = tmp.groupby([drug_col_name, study_id_col], as_index=False)[rig_cols].max()

    # count PMIDs per drug with flag==1
    counts = per_drug_pmid.groupby(drug_col_name, as_index=False)[rig_cols].sum()

    # rename into nice column names (even if original cols differ)
    rename_map = {}
    rename_map[rb_col] = "articles_blinding"
    rename_map[rr_col] = "articles_randomization"
    rename_map[rw_col] = "articles_welfare"
    counts = counts.rename(columns=rename_map)

    out = out.merge(counts, on=drug_col_name, how="left")
    for c in ["articles_blinding", "articles_randomization", "articles_welfare"]:
        if c not in out.columns:
            out[c] = 0
        out[c] = out[c].fillna(0).astype(int)

    return out
    
def get_heterogeneity_stats(df, agg_dict, drug_col_name, study_id_col="PMID"):

    table_by_drug = (df
      .groupby(drug_col_name, dropna=False)
      .agg(**agg_dict)
      .reset_index()
      .rename(columns={drug_col_name: 'drug_name'}))

    # ---- add article-level rigor counts per drug ----
    rigor_by_drug = rigor_article_counts_by_entity(
        df=df,
        drug_col_name=drug_col_name,
        pick_rig_cols_fn=pick_rig_cols,
        study_id_col=study_id_col
    ).rename(columns={drug_col_name: "drug_name"})

    table_by_drug = table_by_drug.merge(rigor_by_drug, on="drug_name", how="left")
    # ------------------------------------------------------

    per_drug_flags = table_by_drug.apply(compute_flags, axis=1)
    total_drugs = len(table_by_drug)

    aggregate_stats = pd.Series({
        'n_entities_in_table': total_drugs,
        'tested_in_both_sexes': fmt_count_pct(int(per_drug_flags['tested_both_sexes'].sum()), total_drugs),
        'tested_in_>=2_species': fmt_count_pct(int(per_drug_flags['species_ge2'].sum()), total_drugs),
        'tested_in_>=2_strains': fmt_count_pct(int(per_drug_flags['strains_ge2'].sum()), total_drugs),
        'tested_with_>=2_outcomes': fmt_count_pct(int(per_drug_flags['assays_ge2'].sum()), total_drugs),
        'tested_in_>=2_countries': fmt_count_pct(int(per_drug_flags['countries_ge2'].sum()), total_drugs),
        "blinding_reported_ge1_article": fmt_count_pct(int(per_drug_flags["blinding_ge1_article"].sum()), total_drugs),
        "randomization_reported_ge1_article": fmt_count_pct(int(per_drug_flags["randomization_ge1_article"].sum()), total_drugs),
        "welfare_reported_ge1_article": fmt_count_pct(int(per_drug_flags["welfare_ge1_article"].sum()), total_drugs),
    })

    aggregate_stats.loc['total_number_of_studies'] = int(df[study_id_col].nunique())

    return aggregate_stats, table_by_drug, per_drug_flags

def compute_experimental_chars(df, agg_dict, drug_col="unique_drug_target"):

    # FAILED DRUGS
    non_approved_studies = df[(df['study_classification'] != 'approved')].copy()
    print(f"Size of failed ds {non_approved_studies.shape} with {non_approved_studies[drug_col].nunique()} drugs and {non_approved_studies['PMID'].nunique()} studies")
    aggregate_stats_non_approved, table_by_drug_fail, per_drug_flags_fail = get_heterogeneity_stats(non_approved_studies, agg_dict, drug_col_name=drug_col)
    
    filed_drugs = non_approved_studies.merged_umls_label.nunique()
    failed_diseases = non_approved_studies.merged_mondo_label.nunique()
    aggregate_stats_non_approved.loc["n_unique_drugs"] = int(filed_drugs)
    aggregate_stats_non_approved.loc["n_unique_diseases"] = int(failed_diseases)
    
    # ROB
    failed_rob = rob_percents_per_subset(non_approved_studies, pick_rig_cols)
    aggregate_stats_non_approved.loc['%_rigor_blinding']     = failed_rob['%_rigor_blinding']
    aggregate_stats_non_approved.loc['%_rigor_randomization'] = failed_rob['%_rigor_randomization']
    aggregate_stats_non_approved.loc['%_rigor_welfare']       = failed_rob['%_rigor_welfare']


    # SUCCESFULL DRUGS
    approved_matches = df[(df['study_classification'] == 'approved')].copy()
    print(f"Size of approved ds {approved_matches.shape} with {approved_matches[drug_col].nunique()} {drug_col} and {approved_matches['PMID'].nunique()} studies")
    aggregate_stats_approved, table_by_drug_appr, per_drug_flags_appr = get_heterogeneity_stats(approved_matches, agg_dict, drug_col_name=drug_col)
    approved_drugs = approved_matches.merged_umls_label.nunique()
    approved_diseases = approved_matches.merged_mondo_label.nunique()
    aggregate_stats_approved.loc["n_unique_drugs"] = int(approved_drugs)
    aggregate_stats_approved.loc["n_unique_diseases"] = int(approved_diseases)

    # ROB
    approved_rob = rob_percents_per_subset(approved_matches, pick_rig_cols)
    aggregate_stats_approved.loc['%_rigor_blinding']     = approved_rob['%_rigor_blinding']
    aggregate_stats_approved.loc['%_rigor_randomization'] = approved_rob['%_rigor_randomization']
    aggregate_stats_approved.loc['%_rigor_welfare']       = approved_rob['%_rigor_welfare']

    # -----------------------------
    # Side-by-side comparison (no .T)
    # -----------------------------
    comparison_df = pd.DataFrame({
    'Approved': aggregate_stats_approved,
    'Non-approved': aggregate_stats_non_approved
})

    # rows for which we want the delta (pair-level + optionally study-level)
    delta_rows = [
        'tested_in_both_sexes', 'tested_in_>=2_species', 'tested_in_>=2_strains',
        'tested_with_>=2_outcomes', 'tested_in_>=2_countries',
        'blinding_reported_ge1_article', 'randomization_reported_ge1_article',
        'welfare_reported_ge1_article',
    
        # OPTIONAL: keep study-level deltas too; remove these 3 if you don't want them
        '%_rigor_blinding', '%_rigor_randomization', '%_rigor_welfare',
    ]
    delta_rows = [r for r in delta_rows if r in comparison_df.index]
    
    pct_only = comparison_df.loc[delta_rows].applymap(to_pct)
    
    comparison_df.loc[delta_rows, 'Δ % (Apprv - Non-apprv)'] = (
        pct_only['Approved'] - pct_only['Non-approved']
    )
    
    comparison_df.loc[delta_rows, 'Δ % (Apprv - Non-apprv)'] = (
        comparison_df.loc[delta_rows, 'Δ % (Apprv - Non-apprv)']
        .apply(lambda x: f"{x:+.1f} %" if pd.notna(x) else "")
    )
    
    # tidy row order
    order = [
        "n_entities_in_table",
        "total_number_of_studies",
        "n_unique_drugs",
        "n_unique_diseases",
    
        'tested_in_both_sexes', 'tested_in_>=2_species', 'tested_in_>=2_strains',
        'tested_with_>=2_outcomes', 'tested_in_>=2_countries',
    
        # pair-level rigor rows 
        "Rigor: blinding reported (pairs)",
        "Rigor: randomization reported (pairs)",
        "Rigor: welfare reported (pairs)",
    
        # OPTIONAL: study-level rigor rows
        '%_rigor_blinding', '%_rigor_randomization', '%_rigor_welfare',
    ]
    existing = [r for r in order if r in comparison_df.index]
    comparison_df = comparison_df.loc[existing + [i for i in comparison_df.index if i not in existing]]

    return aggregate_stats_non_approved, aggregate_stats_approved, comparison_df, table_by_drug_fail, table_by_drug_appr, per_drug_flags_fail, per_drug_flags_appr




In [ ]:
rigor_cols = ['rigor_blinding', 'rigor_randomization', 'rigor_welfare', 'sample_size']
for col in rigor_cols:
    combined_preclin_df[col + '_binary'] = combined_preclin_df[col].apply(lambda x: 1 if isinstance(x, str) and 'present' in x else 0)

In [ ]:
combined_preclin_df.head()

In [ ]:
combined_preclin_df.shape

In [ ]:
combined_preclin_df[combined_preclin_df['normalized_key']=="traumatic brain injury <> sildenafil"].to_csv("paper_example.csv")

In [ ]:
combined_preclin_df.shape

In [ ]:
combined_preclin_df_gt1 = combined_preclin_df[combined_preclin_df['preclinical_count_before_clinical']>1]
combined_preclin_df_gt1.shape

### table

In [ ]:
agg_dict = {
    'animal_sex': ('animal_sex', unique_concat),
    'animal_strain': ('animal_strain', unique_concat),
    'animal_species': ('animal_species', unique_concat),
    'assay_type': ('assay_type', unique_concat),
    'first_author_country': ('first_author_country', unique_concat),
    'PMID': ('PMID', unique_concat),
}



In [ ]:
combined_preclin_df_more_than_one = combined_preclin_df[combined_preclin_df['preclinical_count_before_clinical']>1]
combined_preclin_df_more_than_one.shape

In [ ]:
aggregate_stats_non_approved, aggregate_stats_approved, comparison_df, table_by_drug_fail, table_by_drug_appr, per_drug_flags_fail, per_drug_flags_appr = compute_experimental_chars(combined_preclin_df_more_than_one, agg_dict, drug_col="normalized_key")

comparison_df

In [ ]:
aggregate_stats_non_approved, aggregate_stats_approved, comparison_df, table_by_drug_fail, table_by_drug_appr, per_drug_flags_fail, per_drug_flags_appr = compute_experimental_chars(combined_preclin_df, agg_dict, drug_col="normalized_key")

comparison_df

In [ ]:
table_by_drug_appr.head()

In [ ]:
table_by_drug_appr[table_by_drug_appr['drug_name']=='epilepsy <> phenobarbital']

In [ ]:
combined_preclin_df_agg = pd.concat([table_by_drug_appr, table_by_drug_fail])
combined_preclin_df_agg.shape

In [ ]:
combined_preclin_df_agg.to_csv(f"out/interm_drug_disease_pairs/drug_disease_pairs_annotations_agg_{suffix}.csv",index=False)

# Prep for Associations Analysis

In [ ]:
failed_entities = pd.concat([table_by_drug_fail[['drug_name']], per_drug_flags_fail], axis=1)
failed_entities['target']=0
failed_entities.head()

In [ ]:
translated_entities = pd.concat([table_by_drug_appr[['drug_name']], per_drug_flags_appr], axis=1)
translated_entities['target']=1

translated_entities.head()

In [ ]:
# Append the translated rows to the failed rows
all_entities = pd.concat([failed_entities, translated_entities], axis=0)

# Highly recommended: Reset the index
all_entities = all_entities.reset_index(drop=True)

In [ ]:
all_entities[all_entities['drug_name']=="traumatic brain injury <> sildenafil"]

In [ ]:
# Identify all columns ending in 'ge2'
cols_to_drop = [c for c in all_entities.columns if (c.endswith('ge2') or c.endswith('ge1_article'))]

# Drop them from the dataframe
all_entities = all_entities.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} redundant columns: {cols_to_drop}")

In [ ]:
all_entities.target.value_counts()

In [ ]:
all_entities[all_entities['species_unique']==8]

In [ ]:
all_entities[all_entities['articles_total']>200]

In [ ]:
list(table_by_drug_appr[table_by_drug_appr['drug_name']=="epilepsy <> diazepam"].animal_species)

In [ ]:
all_entities.head()

In [ ]:
len(all_entities) - len(all_entities[all_entities['articles_total']==1])

#### save for R

In [ ]:
df = all_entities.copy()

suffix = dataset_name.split()[0] 

path_main = f"/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/df_translation_associations_{suffix}.csv"
path_out  = f"out/df_translation_associations_{suffix}.csv"

df.to_csv(path_main, index=False)
df.to_csv(path_out, index=False)

print("Saved files:")
print(path_main)
print(path_out)

### viz

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Select numeric columns (including target)
numeric_df = all_entities.select_dtypes(include=['number'])

# 2. Compute correlation matrix
corr_matrix = numeric_df.corr(method="spearman")

# 3. Focus on correlations with 'target'
target_corr = corr_matrix['target'].sort_values(ascending=False)

print("Correlation of features with Target:")
print(target_corr)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap of Preclinical Metrics and Target')
plt.show()

## Sample data for validation

In [ ]:
sampled_df = pd.concat([
    all_entities[all_entities["target"] == 1].sample(n=5, random_state=42),
    all_entities[all_entities["target"] == 0].sample(n=5, random_state=42)
])
sampled_df

In [ ]:
numeric_cols = [
    "tested_both_sexes",
    "species_unique",
    "strain_unique",
    "assay_unique",
    "country_unique",
]

# compute z-scores
z_scores = (
    all_entities[numeric_cols]
    .apply(lambda x: (x - x.mean()) / x.std())
    .abs()
)

# mark outliers (any column with z > 2.5)
outliers = all_entities[z_scores.max(axis=1) > 2.5]

outlier_sample = outliers.sample(n=5, random_state=42)
outlier_sample

In [ ]:
combined_preclin_df.head()

In [ ]:
sampled_and_outliers_df = pd.concat([
    sampled_df,
    outlier_sample
])
sampled_and_outliers_df

In [ ]:
out_dir="./data/translation_validation"
def safe_filename(s: str) -> str:
    s = s.strip()
    s = re.sub(r"[^\w\-<> ]+", "", s)
    s = s.replace(" <> ", "_")
    s = re.sub(r"\s+", "_", s)
    return s[:200]

# df_drugs must contain: drug_name, target
for _, row in (
    sampled_and_outliers_df[["drug_name", "target"]]
    .dropna()
    .drop_duplicates()
    .iterrows()
):
    key = row["drug_name"]
    target = int(row["target"])

    prefix = "positive_" if target == 1 else "negative_"

    sub = combined_preclin_df[
        combined_preclin_df["normalized_key"] == key
    ].copy()

    if sub.empty:
        continue
        

    if len(sub) > 5:
        sub = sub.sample(n=5, random_state=42)
        
    fname = prefix + safe_filename(key) + ".csv"
    path = os.path.join(out_dir, fname)

    sub.to_csv(path, index=False)

print(f"Saved per-drug CSVs with target prefix to: {out_dir}")

In [ ]:
sampled_and_outliers_df.to_csv(out_dir+"/sampled_and_outliers_df.csv",index=False)

In [ ]:
 combined_preclin_df[
        combined_preclin_df["normalized_key"] == "epilepsy <> phenytoin"
    ].to_csv(out_dir +"/epilepsy_phenytoin_all.csv",index=False)

In [ ]:
combined_preclin_df[
    (combined_preclin_df["normalized_key"] == "focal epilepsy <> carbamazepine") &
    (combined_preclin_df["animal_species"].str.contains("dog"))
]


In [ ]:
combined_preclin_df[
    (combined_preclin_df["normalized_key"] == "focal epilepsy <> carbamazepine")
].animal_species

In [ ]:
combined_preclin_df[combined_preclin_df['PMID']==24878681]

# Viz Drug-Disease Clinical x Preclinical

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

def viz_joined_preclin_clinical(
    filtered_df,
    normalized_key="normalized_condition",
    additional_sort_by="both",
    translation_column=None,
    top_n=25,
    fig_name_suffix="",
    save_to="png",
    save_path="06_preclin_clinic_join/viz",
):
    """
    Visualizes the top N normalized conditions with:
      - Preclinical Count
      - Preclinical Count Before Clinical
      - Clinical Count
    as grouped horizontal bars.

    Adds ♦ and min_relevant_clinical_year next to clinical count if translation_column is True.
    """

    # --- select top N ---
    if additional_sort_by == "both":
        top_n_df = filtered_df.head(top_n)
        title_str = "Clinical + Preclinical Count"
    elif additional_sort_by == "clinical_count":
        top_n_df = filtered_df.sort_values(by="clinical_count", ascending=False).head(top_n)
        title_str = "Clinical Count"
    elif additional_sort_by == "tail":
        top_n_df = filtered_df.tail(top_n)
        title_str = "Clinical + Preclinical Count Tail"
    else:
        top_n_df = filtered_df.sort_values(by="preclinical_count", ascending=False).head(top_n)
        title_str = "Preclinical Count"

    conditions = top_n_df[normalized_key].astype(str)
    clinical_counts = top_n_df["clinical_count"].fillna(0)
    preclinical_counts = top_n_df["preclinical_count"].fillna(0)
    preclinical_before_clinical_counts = top_n_df["preclinical_count_before_clinical"].fillna(0)
    min_years = top_n_df.get("min_relevant_clinical_year")

    y_positions = np.arange(len(conditions))
    bar_width = 0.24

    os.makedirs(save_path, exist_ok=True)

    plt.figure(figsize=(14, 10))

    # Bars
    plt.barh(y_positions - bar_width, preclinical_counts, height=bar_width,
             label="Preclinical Count", color="#56B4E9", zorder=2)

    plt.barh(y_positions, preclinical_before_clinical_counts, height=bar_width,
             label="Preclinical Before Clinical", color="#009E73", zorder=2)

    plt.barh(y_positions + bar_width, clinical_counts, height=bar_width,
             label="Clinical Count", color="#E69F00", zorder=2)

    # Labels
    for i in range(len(conditions)):

        # Preclinical total
        plt.text(preclinical_counts.iloc[i],
                 y_positions[i] - bar_width,
                 f"{preclinical_counts.iloc[i]:.0f}",
                 va="center", fontsize=13)

        # Preclinical before clinical
        plt.text(preclinical_before_clinical_counts.iloc[i],
                 y_positions[i],
                 f"{preclinical_before_clinical_counts.iloc[i]:.0f}",
                 va="center", fontsize=13)

        # Clinical + optional diamond + year
        clinical_text = f"{clinical_counts.iloc[i]:.0f}"

        if translation_column and bool(top_n_df[translation_column].iloc[i]):
            year_str = ""
            if min_years is not None and not pd.isna(min_years.iloc[i]):
                year_str = f" {int(min_years.iloc[i])}"
            clinical_text += f" ♦{year_str}"

        plt.text(clinical_counts.iloc[i],
                 y_positions[i] + bar_width,
                 clinical_text,
                 va="center", fontsize=13)

    # Axis formatting
    plt.yticks(y_positions, conditions)
    plt.xlabel("Study Count", fontsize=15)
    plt.title(f"Top {top_n} Disease-Drug Pairs by {title_str}", fontsize=18)

    if translation_column:
        plt.legend(
            handles=[
                plt.Line2D([0], [0], color="#56B4E9", lw=6, label="Preclinical Count"),
                plt.Line2D([0], [0], color="#009E73", lw=6, label="Preclinical Before Clinical"),
                plt.Line2D([0], [0], color="#E69F00", lw=6, label="Clinical Count"),
                plt.Line2D([0], [0], color="black", marker="D", linestyle="",
                           label="At least one Phase 4 trial (♦ year shown)"),
            ],
            loc="lower right",
            fontsize=12,
        )
    else:
        plt.legend(loc="lower right", fontsize=12)

    plt.tick_params(axis="both", labelsize=12)
    plt.gca().invert_yaxis()
    plt.grid(axis="x", linestyle="--", alpha=0.4, color="gray", zorder=0)

    plt.tight_layout()

    out_file = f"{save_path}/top{top_n}_preclin_clin{fig_name_suffix}.{save_to}"
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
df_translated['translated']=True
df_failed['translated']=False

In [ ]:
df_combined_with_articles = pd.concat([df_failed, df_translated], axis=0, ignore_index=True)
df_combined_with_articles.shape

In [ ]:
df_combined_with_articles["preclinical_count_before_clinical"] = (
    df_combined_with_articles["preclinical_count_before_clinical"]
    .fillna(df_combined_with_articles["preclinical_count_before_latest_trial"])
)


In [ ]:
df_combined_with_articles_to_merge = df_combined_with_articles[['normalized_key','preclinical_count_before_clinical','translated','min_relevant_clinical_year']]
translation_table_for_viz = translation_table.merge(df_combined_with_articles_to_merge, left_on="normalized_key", right_on="normalized_key",how="left")

In [ ]:
translation_table_for_viz.shape

In [ ]:
# ------------------------- #
#         VISUALIZE         #
# ------------------------- #
translation_table_for_viz['key_for_viz'] = (
    translation_table_for_viz['merged_umls_label'] + "\n" + translation_table_for_viz['merged_mondo_label']
)

viz_joined_preclin_clinical(
    translation_table_for_viz,
    "key_for_viz",
    translation_column='translated',
    top_n=10,
    fig_name_suffix='',
    save_path="viz/"
)


In [ ]:
translation_table_for_viz.translated.value_counts()

In [ ]:
translation_table_for_viz["translated"] = (
    translation_table_for_viz["translated"]
        .replace({True: "translated", False: "not translated",
                  "True": "translated", "False": "not translated"})
        .fillna("unknown")
)
plot_df = translation_table_for_viz.dropna(
    subset=["min_relevant_clinical_year", "preclinical_count_before_clinical"]
).copy()
plot_df.translated.value_counts()

In [ ]:
plt.figure(figsize=(10, 6))

# Okabe–Ito colorblind-safe palette
color_map = {
    "translated": "#0072B2",      # blue
    "not translated": "#E69F00",  # orange
    "unknown": "#999999"          # neutral grey
}

# Fixed legend order
for status in ["translated", "not translated", "unknown"]:
    subset = plot_df[plot_df["translated"] == status]

    if len(subset) == 0:
        continue

    plt.scatter(
        subset["min_relevant_clinical_year"],
        subset["preclinical_count_before_clinical"],
        color=color_map[status],
        edgecolor="black",
        linewidth=0.5,
        alpha=0.85,
        s=75,
        label=status.capitalize()
    )

plt.xlabel("Minimum Relevant Clinical Year", fontsize=13)
plt.ylabel("Preclinical Count Before Clinical", fontsize=13)
plt.title("Preclinical Evidence vs First Relevant Clinical Year", fontsize=15)

plt.legend(frameon=False)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
len(failed_pmids)

In [ ]:
len(translated_pmids)

#### Pub years per group

In [ ]:
table_by_drug_appr.articles_total.sum()

In [ ]:
pmids_translated = (
    table_by_drug_appr['PMID']
    .dropna()
    .astype(str)
    .str.split(r',\s*')
    .explode()
)

pmids_translated = pd.to_numeric(pmids_translated, errors='coerce')
pmids_translated = pmids_translated.dropna().astype(int).unique()

len(pmids_translated)

In [ ]:
pmids_failed = (
    table_by_drug_fail['PMID']
    .dropna()
    .astype(str)
    .str.split(r',\s*')
    .explode()
)

pmids_failed = pd.to_numeric(pmids_failed, errors='coerce')
pmids_failed = pmids_failed.dropna().astype(int).unique()

len(pmids_failed)

In [ ]:
preclin_dataset_year = pd.read_csv(df_path_current_dataset)[['PMID', 'year']]#, 'rigor_randomization', 'rigor_welfare', 'sample_size', 'animal_species','animal_sex','animal_strain','animal_number', 'assay_type', 'first_author_country']]

In [ ]:
translated_df_years = preclin_dataset_year[preclin_dataset_year["PMID"].isin(pmids_translated)][["PMID", "year"]].copy()
failed_df_years = preclin_dataset_year[preclin_dataset_year["PMID"].isin(pmids_failed)][["PMID", "year"]].copy()


translated_df_years["status"] = "translated"
failed_df_years["status"] = "failed"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plot_df = pd.concat([translated_df_years, failed_df_years], ignore_index=True)

plt.figure()

plt.hist(
    plot_df.loc[plot_df["status"] == "translated", "year"],
    bins=20,
    alpha=0.5,
    label="translated"
)

plt.hist(
    plot_df.loc[plot_df["status"] == "failed", "year"],
    bins=20,
    alpha=0.5,
    label="failed"
)

plt.xlabel("Year")
plt.ylabel("Number of studies")
plt.title("Distribution of Publication Years")
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = pd.concat([translated_df_years, failed_df_years], ignore_index=True)
plot_df["year"] = pd.to_numeric(plot_df["year"], errors="coerce")
plot_df = plot_df.dropna(subset=["year", "status"]).copy()
plot_df["status"] = plot_df["status"].str.capitalize()
plot_df["status"] = plot_df["status"].replace({"Failed": "Non-translated"})
order = ["Non-translated", "Translated"]
sns.set_theme(style="whitegrid", context="talk")

plt.figure(figsize=(6.8, 5.5))

sns.violinplot(
    data=plot_df,
    x="status",
    y="year",
    order=order,
    cut=0,
    linewidth=1.2,
    inner=None
)

sns.boxplot(
    data=plot_df,
    x="status",
    y="year",
    order=order,
    width=0.2,
    showfliers=False,
    boxprops={"facecolor": "none", "edgecolor": "black", "zorder": 3},
    medianprops={"color": "black", "linewidth": 2.5},
    whiskerprops={"linewidth": 1.2},
    capprops={"linewidth": 1.2}
)

plt.xlabel("")
plt.ylabel("Publication year")
plt.title("Publication Year of Included Animal Studies", fontsize=17, pad=10)
sns.despine()
plt.tight_layout()
out_file = f"viz/pub_year_distribution_{dataset_name.replace(' ', '_')}.pdf"
plt.savefig(out_file, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
summary = (
    plot_df
    .groupby("status")["year"]
    .agg(
        n="count",
        median="median",
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75)
    )
    .loc[["Non-translated", "Translated"]]  # enforce order
)
summary